# ChaoSSM - Phase 16: the Mackey-Glass pilot, re-run under the canonical protocol

**Why.** The real-analog pilot in the manuscript predates two corrections and is now the only result in
the paper on a different footing from everything else:

1. it trained the scalar input encoder alongside the readout, so it reports 2,336 trainable parameters
   where the rest of the paper reports 2,208 under the canonical readout-only protocol; and
2. it tested only the edge arm, so it cannot speak to whether the benefit is specific to criticality -
   the question the regime-pure controls (Phase 14/15) answer in the negative.

This notebook fixes both: frozen input encoder (canonical), and an **ordered** arm alongside edge and
deep chaos, on both the Mackey-Glass series and the non-chaotic control series, at n = 5 seeds.

**What it should settle.** If the ordered arm matches or beats the edge arm on Mackey-Glass, the
real-analog evidence lines up with the six synthetic systems and the pilot can be reported on the same
protocol as everything else. The non-chaotic control series is retained unchanged: it tests whether the
benefit depends on the *target* being chaotic, which is a separate question from the reservoir's regime
and remains interesting either way.

**Cost.** 2 series x (3 regimes x 5 seeds + 5 linear) = 40 runs at 2,500 steps, elementwise map with a
frozen encoder, so this is the cheap end - expect roughly 20-40 min on a T4.

**Outputs** go to `results/phase16_mackeyglass_canonical/`, touching no earlier phase. The run is
resumable (per-run checkpointing) and zips its results if Drive is unavailable.

## Setup

In [ ]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict, field
from itertools import product
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))
def setup_results_dir(project="ChaoSSM",phase="phase16_mackeyglass_canonical"):
    """Resolve the output directory, preferring Google Drive so results survive the runtime.

    Colab's drive.mount() can fail (e.g. ValueError when /content/drive already holds files, or
    when the mount is refused). Earlier versions silently fell back to the VM's local disk, where
    results are lost when the runtime recycles. We now retry the mount once with force_remount and,
    if Drive is still unavailable, fall back to local disk with a loud warning."""
    base=None
    try:
        from google.colab import drive
        try:
            drive.mount("/content/drive",force_remount=False)
        except Exception as e1:
            print(f"[warn] drive.mount failed ({type(e1).__name__}: {e1}); retrying with force_remount=True")
            drive.mount("/content/drive",force_remount=True)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"/content/{project}_results/{phase}") if os.path.isdir("/content") \
             else os.path.abspath(f"./{project}_results/{phase}")
        print("="*78)
        print(f"[WARNING] Google Drive is NOT mounted ({type(e).__name__}).")
        print(f"[WARNING] Results will be written to {base}")
        print("[WARNING] That is the runtime's local disk -- the files are LOST when the runtime")
        print("[WARNING] recycles. Download them, or copy them to Drive, as soon as the run ends.")
        print("="*78)
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
# NOTE: this notebook writes ONLY to its own results/phase16_mackeyglass_canonical/ folder.
# It creates new phase14_*.csv / .png files and does NOT touch or overwrite any existing
# Phase 10/13/14 (or other phase) results. Nothing prior is modified.
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

## Configuration

In [ ]:
SMOKE_TEST=False   # True runs a ~2 min code check

@dataclass
class Config:
    state_dim:int=64; d_model:int=64
    steps:int=2500; batch:int=64; warmup_frac:float=0.05; lr_min_frac:float=0.1
    grad_clip:float=1.0; gamma_init:float=1.0; lr:float=1e-3
    seeds:int=5
    eval_n:int=1500; eval_seed:int=12345
    seq_len:int=64; fc_bins:int=32
    r_cap_map:float=4.0
    # the ordered target is the addition that makes this comparable to Phase 14/15
    lambda_target:dict=field(default_factory=lambda: dict(ordered=-0.30, edge=-0.02, deep=0.30))
    lambda_tol:float=0.05; calib_steps:int=200; calib_warm:int=60
    chaos_regimes:tuple=("ordered","edge","deep")
    series:tuple=("mackey_glass","nonchaotic_control")

CFG=Config()
if SMOKE_TEST:
    CFG.steps=30; CFG.seeds=2; CFG.eval_n=40; CFG.calib_steps=30; CFG.calib_warm=10
    CFG.chaos_regimes=("ordered","edge")
    print("*** SMOKE TEST - results meaningless, this only checks the code runs ***")
print("regimes:",CFG.chaos_regimes,"| series:",CFG.series)
print("runs =",len(CFG.series)*(len(CFG.chaos_regimes)*CFG.seeds+CFG.seeds))

## Cells and gamma-calibration (verbatim from Phase 14/15)

In [ ]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)

class ChaosCell(nn.Module):
    def __init__(self,cfg,regime):
        super().__init__(); self.N=cfg.state_dim; self.regime=regime
        self.r_cap=float(cfg.r_cap_map)
        self.w_r=nn.Parameter(torch.zeros(self.N))
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(cfg.gamma_init)))))
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def set_gamma(self,val):
        with torch.no_grad(): self.w_gamma.copy_(torch.tensor(float(math.log(math.expm1(max(val,1e-3))))))
    def phi(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return triangle_fold(self.get_gamma()*u)
    def step(self,h,x): return triangle_fold(self.phi(h)+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
    def freeze_recurrence(self):
        for p_ in [self.w_r,self.W_mix,self.w_gamma]: p_.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class LinearCell(nn.Module):
    def __init__(self,cfg,**kw):
        super().__init__(); self.N=cfg.state_dim
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_lin=nn.Parameter(W*0.9)
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def step(self,h,x): return torch.tanh(h@self.W_lin.T+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): self.W_lin.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class GRUCellArm(nn.Module):
    def __init__(self,cfg,**kw): super().__init__(); self.N=cfg.state_dim; self.cell=nn.GRUCell(cfg.d_model,cfg.state_dim)
    def step(self,h,x): return self.cell(x,h)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): pass
    def freeze_input(self): pass

def make_cell(cfg,cond):
    if cond["arm"]=="gru": return GRUCellArm(cfg)
    if cond["arm"]=="linear": return LinearCell(cfg)
    return ChaosCell(cfg,cond["regime"])

In [ ]:
from torch.func import jacrev
@torch.no_grad()
def map_lyapunov(cell,cfg,seed=0,steps=None,warm=None):
    steps=steps or cfg.calib_steps; warm=warm or cfg.calib_warm; N=cfg.state_dim
    dev=cell.W_mix.device
    g=torch.Generator(device=dev); g.manual_seed(seed)
    h=torch.full((N,),0.5,device=dev); Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum=torch.zeros(N,device=dev)
    for t in range(steps+warm):
        def stp(z):
            r=cell.get_r(); u=z@cell.W_mix.T+r*z*(1-z); return triangle_fold(cell.get_gamma()*u)
        J=jacrev(stp)(h); h=stp(h)+0.02*torch.randn(N,generator=g,device=dev); h=torch.clamp(h,0.0,1.0)
        Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
        if t>=warm: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
    return float(torch.sort(logsum/steps,descending=True)[0][0])

def calibrate_gamma(cell,cfg,regime,seed=0):
    # lo widened from Phase 14's 0.05 to 0.01: the most ordered target (-0.60) needs a smaller gain
    # than any Phase 14 target did. Returns a calibration-success flag so that points the cell
    # cannot actually reach are reported as such rather than silently mislabeled.
    target=cfg.lambda_target[regime]; lo,hi=0.01,8.0; best=None; best_err=1e9
    for _ in range(18):
        mid=math.sqrt(lo*hi); cell.set_gamma(mid); lam=map_lyapunov(cell,cfg,seed=seed)
        err=abs(lam-target)
        if err<best_err: best_err=err; best=(mid,lam)
        if lam<target: lo=mid
        else: hi=mid
        if err<cfg.lambda_tol: break
    cell.set_gamma(best[0]); return best[0],best[1],bool(best_err<cfg.lambda_tol)


## The two series (verbatim from the original pilot)

Mackey-Glass at the standard $\tau=17$, and a non-chaotic control built from incommensurate sinusoids
plus mild noise. Both are min-max normalized to $[0,1]$ and discretized into 32 bins exactly as the
synthetic forecasting task is, so accuracies are on the same footing.

In [ ]:
def mackey_glass_series(n, tau=17, beta=0.2, gamma=0.1, nexp=10, dt=1.0, seed=0, discard=500):
    rng=np.random.default_rng(seed); hist=int(tau/dt)+1
    x=np.zeros(n+hist+discard); x[:hist]=1.2+0.2*rng.standard_normal(hist)
    for t in range(hist,len(x)):
        xt=x[t-hist]; x[t]=x[t-1]+dt*(beta*xt/(1+xt**nexp)-gamma*x[t-1])
    return x[hist+discard:]

def nonchaotic_series(n, seed=0):
    rng=np.random.default_rng(seed); t=np.arange(n)
    freqs=[0.017,0.041,0.073]; phases=rng.uniform(0,2*np.pi,len(freqs))
    return sum(np.sin(2*np.pi*f*t+p) for f,p in zip(freqs,phases))+0.15*rng.standard_normal(n)

def normalize01(x):
    lo,hi=x.min(),x.max(); return (x-lo)/(hi-lo+1e-9)

_SRC={}
def get_source(name, seed, length=60000):
    if (name,seed) not in _SRC:
        raw=mackey_glass_series(length,seed=seed) if name=="mackey_glass" else nonchaotic_series(length,seed=seed)
        _SRC[(name,seed)]=normalize01(raw)
    return _SRC[(name,seed)]

def make_forecast_real(cfg,batch,device,gen,series_name,src_seed):
    L=cfg.seq_len; src=get_source(series_name,src_seed); n=len(src)
    # NOTE: torch.randint allocates on CPU by default, so passing a CUDA generator raises
    # "Expected a 'cpu' device type for generator but found 'cuda'". Allocate on the
    # generator's own device so this works identically on CPU and GPU runtimes.
    g=np.random.default_rng(int(torch.randint(0,2**31-1,(1,),generator=gen,device=gen.device).item()))
    starts=g.integers(0,n-L-1,size=batch)
    X=torch.tensor(np.stack([src[s:s+L+1] for s in starts]),dtype=torch.float32,device=device)
    inp=X[:,:L]; tgt=torch.clamp((X[:,1:L+1]*cfg.fc_bins).long(),0,cfg.fc_bins-1)
    return inp,tgt

def majority_chance(cfg,name,src_seed):
    src=get_source(name,src_seed); b=np.clip((src[1:]*cfg.fc_bins).astype(int),0,cfg.fc_bins-1)
    cnt=np.bincount(b,minlength=cfg.fc_bins); return float(cnt.max())/float(cnt.sum())

for nm in CFG.series:
    s=get_source(nm,0,5000)
    print(f"{nm:20s} range=[{s.min():.3f},{s.max():.3f}] std={s.std():.3f} "
          f"majority-class chance={majority_chance(CFG,nm,0):.4f}")

## Readout model - canonical, frozen input encoder

This is the Phase 13d `ReservoirLM`: the encoder is frozen so the entire input pathway is fixed and
random, the reservoir is rolled out under `torch.no_grad()`, and only the layer normalization and the
linear readout train. `n_trainable` is recorded per run and should read 2,208 throughout, matching the
single-system fair test.

In [ ]:
class ReservoirLM(nn.Module):
    def __init__(self,cfg,cell,out_dim):
        super().__init__(); self.inp=nn.Linear(1,cfg.d_model); self.cell=cell
        self.ln=nn.LayerNorm(cfg.state_dim); self.readout=nn.Linear(cfg.state_dim,out_dim)
        for p_ in self.inp.parameters(): p_.requires_grad_(False)   # canonical: random, frozen drive
    def forward(self,x):
        with torch.no_grad():
            e=self.inp(x.unsqueeze(-1)); B,T=e.shape[0],e.shape[1]
            h=self.cell.init_state(B,e.device,e.dtype); H=[]
            for t in range(T):
                h=self.cell.step(h,e[:,t]); H.append(h)
            H=torch.stack(H,1)
        return self.readout(self.ln(H))

def make_lr_lambda(cfg,total):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn

@torch.no_grad()
def eval_acc_real(model,cfg,series_name,src_seed,chunk=256):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(cfg.eval_seed)
    inp,tgt=make_forecast_real(cfg,cfg.eval_n,DEVICE,g,series_name,src_seed)
    model.eval(); c=tot=0.0
    for i in range(0,inp.shape[0],chunk):
        p=model(inp[i:i+chunk]).argmax(-1); t=tgt[i:i+chunk]
        c+=(p==t).float().sum().item(); tot+=t.numel()
    model.train(); return c/max(tot,1)

## Run (resumable; checkpoints after every run)

In [ ]:
FRESH_START=False
CKPT=os.path.join(RESULTS,"phase16_mackeyglass.csv")
COLS=["series","arm","regime","seed","diverged","acc","calibrated_lambda","calibration_ok","n_trainable"]

if FRESH_START and os.path.exists(CKPT):
    os.replace(CKPT,CKPT+".bak"); print("[resume] FRESH_START -> previous CSV moved to",CKPT+".bak")
if os.path.exists(CKPT):
    rows=pd.read_csv(CKPT).to_dict("records")
    done={(r["series"],r["regime"],int(r["seed"])) for r in rows}
    print(f"[resume] found {len(rows)} completed runs")
else:
    rows=[]; done=set(); print("[resume] no checkpoint -- starting from scratch")

def _flush(): pd.DataFrame(rows)[COLS].to_csv(CKPT,index=False)

planned=[]
for ser in CFG.series:
    for reg in CFG.chaos_regimes: planned += [(ser,reg,sd) for sd in range(CFG.seeds)]
    planned += [(ser,"linear-control",sd) for sd in range(CFG.seeds)]
todo=[k for k in planned if k not in done]
print(f"[resume] {len(planned)} planned, {len(planned)-len(todo)} done, {len(todo)} to run\n")

t0=time.time()
for series,regime,sd in todo:
    torch.manual_seed(sd); np.random.seed(sd)
    gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(sd)
    lam=float("nan"); ok=True
    if regime=="linear-control":
        cell=LinearCell(CFG).to(DEVICE)
    else:
        cell=ChaosCell(CFG,regime).to(DEVICE)
        _,lam,ok=calibrate_gamma(cell,CFG,regime,seed=sd)
        if not ok: print(f"    [calibration WARNING] {series}/{regime} s={sd}: lambda={lam:+.3f} missed target")
    cell.freeze_recurrence(); cell.freeze_input()
    model=ReservoirLM(CFG,cell,CFG.fc_bins).to(DEVICE)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(params,lr=CFG.lr,betas=(0.9,0.95))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(CFG,CFG.steps))
    div=False
    for step in range(CFG.steps):
        inp,tgt=make_forecast_real(CFG,CFG.batch,DEVICE,gen,series,sd)
        lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1))
        opt.zero_grad(set_to_none=True); L.backward()
        gn=torch.nn.utils.clip_grad_norm_(params,CFG.grad_clip).item(); opt.step(); sched.step()
        if not (math.isfinite(L.item()) and math.isfinite(gn)): div=True; break
    acc=eval_acc_real(model,CFG,series,sd) if not div else float("nan")
    rows.append(dict(series=series,arm=("linear" if regime=="linear-control" else "chaos"),
                     regime=regime,seed=sd,diverged=div,acc=acc,
                     calibrated_lambda=lam,calibration_ok=ok,
                     n_trainable=sum(p.numel() for p in params)))
    _flush()
    lt="" if regime=="linear-control" else f" lam={lam:+.3f}"
    print(f"[P16] {series:20s} {regime:14s} s={sd} acc={acc:.4f}{lt}"+(" DIVERGED" if div else ""))

RES=pd.DataFrame(rows)[COLS]; _flush()
print(f"\nwall clock this session: {(time.time()-t0)/60:.1f} min | total runs: {len(RES)}")
print("n_trainable values:",sorted(RES.n_trainable.unique()))

## Analysis

In [ ]:
def ci(x,seed=0):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    if len(x)<2: return (float("nan"),float("nan"))
    rng=np.random.default_rng(seed); b=rng.choice(x,(10000,len(x))).mean(1)
    return float(np.percentile(b,2.5)),float(np.percentile(b,97.5))

out=[]
for ser in CFG.series:
    d=RES[RES.series==ser]
    lin=d[d.regime=="linear-control"].sort_values("seed").acc.values
    for reg in CFG.chaos_regimes:
        a=d[d.regime==reg].sort_values("seed").acc.values
        ndiv=int(d[d.regime==reg].diverged.sum())
        if np.isfinite(a).sum()<2:
            out.append(dict(series=ser,regime=reg,mean_acc=float("nan"),acc_ci="--",
                            vs_linear=float("nan"),vs_linear_ci="--",wins=0,diverged=ndiv)); continue
        k=min(len(a),len(lin)); p=a[:k]-lin[:k]
        lo,hi=ci(a); plo,phi=ci(p,1)
        out.append(dict(series=ser,regime=reg,mean_acc=round(float(np.nanmean(a)),4),
                        acc_ci=f"[{lo:.3f},{hi:.3f}]",
                        vs_linear=round(float(np.nanmean(p)),4),
                        vs_linear_ci=f"[{plo:+.3f},{phi:+.3f}]",
                        wins=int(np.nansum(a[:k]>lin[:k])),diverged=ndiv))
    lo,hi=ci(lin)
    out.append(dict(series=ser,regime="linear",mean_acc=round(float(lin.mean()),4),
                    acc_ci=f"[{lo:.3f},{hi:.3f}]",vs_linear=0.0,vs_linear_ci="--",wins=0,diverged=0))
S=pd.DataFrame(out); print(S.to_string(index=False)); savetable(S,"phase16_summary.csv")

print("\n--- ordered vs edge, paired by seed (the question this re-run adds) ---")
for ser in CFG.series:
    d=RES[RES.series==ser]
    o=d[d.regime=="ordered"].sort_values("seed").acc.values
    e=d[d.regime=="edge"].sort_values("seed").acc.values
    if np.isfinite(o).sum()<2 or np.isfinite(e).sum()<2: print(f"  {ser}: insufficient finite runs"); continue
    k=min(len(o),len(e)); p=e[:k]-o[:k]; lo,hi=ci(p,2)
    print(f"  {ser:20s} ordered={np.nanmean(o):.4f} edge={np.nanmean(e):.4f} "
          f"edge-ordered={np.nanmean(p):+.4f} CI[{lo:+.3f},{hi:+.3f}] ordered better {int(np.nansum(p<0))}/{k}")
print("\nreading: ordered >= edge on Mackey-Glass would put the real-analog pilot on the same footing")
print("as the six synthetic systems; a target-dependent contrast between the two series remains")
print("meaningful regardless, since it concerns the target rather than the reservoir regime.")

## Figure (300 DPI)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.6),sharey=True)
names={"mackey_glass":"Mackey--Glass ($\\tau=17$)","nonchaotic_control":"Non-chaotic control"}
cols={"ordered":"#4C72B0","edge":"#2E7D32","deep":"#B71C1C","linear-control":"#9E9E9E"}
labs={"ordered":"ordered","edge":"edge","deep":"deep chaos","linear-control":"linear"}
for ax,ser in zip(axes,CFG.series):
    d=RES[RES.series==ser]; keys=list(CFG.chaos_regimes)+["linear-control"]
    m=[np.nanmean(d[d.regime==k].acc.values) for k in keys]
    sd=[np.nanstd(d[d.regime==k].acc.values) for k in keys]
    ax.bar(range(len(keys)),m,yerr=sd,capsize=4,color=[cols[k] for k in keys])
    ax.axhline(np.nanmean(d[d.regime=="linear-control"].acc.values),ls="--",color="#777",lw=1.2)
    ax.set_xticks(range(len(keys))); ax.set_xticklabels([labs[k] for k in keys],fontsize=10)
    ax.set_title(names[ser],fontsize=12); ax.grid(axis="y",alpha=.25)
axes[0].set_ylabel("Forecasting accuracy\n(frozen reservoir, canonical protocol)")
fig.suptitle(f"Real-analog pilot under the canonical protocol (mean $\\pm$ sd, n = {CFG.seeds} seeds)",fontsize=12)
fig.tight_layout(); savefig(fig,"phase16_mackeyglass.png"); plt.close(fig); print("figure saved")

## Rescue the results if Drive was not mounted

In [ ]:
import shutil, os
if not RESULTS.startswith("/content/drive"):
    _zip = shutil.make_archive("/content/phase16_results","zip",RESULTS)
    print("\n"+"="*78)
    print("Drive was not mounted, so these results live only on this runtime's local disk.")
    print("A zip has been written to:", _zip)
    print("Download it now before the runtime recycles:")
    print("  from google.colab import files")
    print(f"  files.download('{_zip}')")
    print("="*78)
else:
    print("\nresults are on Drive:", RESULTS)